# Tutorial: Layer 6 SSG Fit Ranking


## 1. 목적

            6번 레이어는 **SSG fit ranking**이다.

            단, 현재 결과물은 공개 추천 리스트가 아니다.

            > 후보명을 공개하기 전, 어떤 후보군이 수동 검토/출처 보강/시장 관찰/보류에 들어가는지 정하는 잠금된 리뷰 퍼널이다.

            ## 사용한 모델/방법

            - multi-objective score: SSG fit, KBO translation, market realism, tool/process, surplus access, failure resilience
            - sensitivity analysis: 가중치를 바꿔도 살아남는지 확인
            - stage gate ranking: 순위 공개 전 수동 검토 단계를 배정
            - locked release policy: 후보명/정확 점수/정확 순위/추천 라벨 비공개


## 2. 최종 점수의 사고방식

            최종 점수는 단순히 좋은 스탯의 평균이 아니다.

            ```text
            FinalScore =
                SSGFit
              + KBOTranslation
              + MarketRealism
              + ToolProcess
              + SurplusAccess
              + FailureResilience
            ```

            그리고 hard gate를 통과하지 못하면, 점수가 높아도 공개 추천으로 가지 않는다.


In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = ROOT.parent

OUT = ROOT / "outputs" / "tables"

def read_table(filename):
    path = OUT / filename
    if not path.exists():
        print(f"[missing] {path}")
        return pd.DataFrame()
    return pd.read_csv(path)

def take_cols(df, cols, n=10):
    keep = [c for c in cols if c in df.columns]
    if not keep:
        return df.head(n)
    return df.loc[:, keep].head(n)

def count_by(df, cols):
    keep = [c for c in cols if c in df.columns]
    if not keep or df.empty:
        return pd.DataFrame()
    return df.groupby(keep, dropna=False).size().reset_index(name="rows").sort_values("rows", ascending=False)

def assert_candidate_names_locked(df):
    sensitive_cols = {"player_name", "search_name", "team_or_org", "player_id"}
    exposed = sensitive_cols.intersection(df.columns)
    if exposed:
        print(f"Candidate-sensitive columns exist but are not displayed here: {sorted(exposed)}")


In [ ]:
gate = read_table("recruitment_gate_status_v33.csv")
take_cols(gate[gate["gate"].eq("G6")], ["gate", "layer", "progress_pct", "status", "decision", "blocking_gap"])


In [ ]:
stage_summary = read_table("layer6_fit_ranking_v0_3_stage_summary_v0_1.csv")
take_cols(stage_summary, ["fit_slot", "ranking_stage_gate_v0_3", "locked_rows", "release_allowed"], n=20)


In [ ]:
stage_pivot = stage_summary.pivot_table(
    index="fit_slot",
    columns="ranking_stage_gate_v0_3",
    values="locked_rows",
    aggfunc="sum",
    fill_value=0,
)
stage_pivot


## 3. sensitivity band

            특정 가중치에서만 좋아 보이는 후보는 위험하다.

            그래서 여러 가중치 조합에서 순위가 얼마나 흔들리는지 보고, 먼저 리뷰할 lane을 나눈다.


In [ ]:
packet = read_table("ssg_fit_source_fill_packet_v0_1.csv")
assert_candidate_names_locked(packet)
take_cols(
    count_by(packet, ["fit_slot", "sensitivity_band", "fit_review_lane"]),
    ["fit_slot", "sensitivity_band", "fit_review_lane", "rows"],
    n=30,
)


## 4. 왜 아직 추천 리스트가 아닌가

            release_allowed가 False라는 것은 모델이 망했다는 뜻이 아니다.

            오히려 교수님 앞에서는 다음 논리가 더 강하다.

            - 모델은 후보를 좁히는 데 성공했다.
            - 하지만 계약/의료/한국행 의사/출처 근거가 잠겨 있으므로 공개 추천은 보류한다.
            - 따라서 현재 산출물은 "최종 답"이 아니라 "현장 검토용 우선순위 큐"다.


In [ ]:
release_check = stage_summary.groupby("release_allowed", dropna=False)["locked_rows"].sum().reset_index()
release_check


## 5. 발표용 한 줄

            6번 레이어는 후보를 무리하게 발표하지 않고, SSG fit이 높은 후보군을 수동 검토 큐로 정렬한 단계다. 이게 현장 스카우팅 흐름과 가장 비슷한 최종 형태다.

            ## 연습문제

            `manual_review_candidate_locked`와 `source_fill_before_rank_locked`의 차이를 설명해보자. 어떤 경우에 팀원이 먼저 기사를 찾아야 할까?
